# nb01 — Generate the ReliefForge dataset (Kaggle, CPU)

Run this notebook with **Settings → Accelerator: none (CPU)** and **Internet: on**.
It clones the repo, generates the full 4,000-scene geometry dataset, runs the QC gate,
and saves everything to `/kaggle/working`. After it finishes: **Save Version → Save & Run All**,then attach the output as a Kaggle Dataset so `nb02_train` can use it without regenerating.

Expected time: ~10–15 min with 4 workers.

In [ ]:
!nvidia-smi 2>/dev/null || echo "no GPU (this notebook is CPU-only - that is fine)"
!nproc && free -g | head -2 && df -h /kaggle/working | tail -1

In [ ]:
%%bash
cd /kaggle/working
rm -rf reliefforge
git clone --depth 1 https://github.com/<YOUR_USERNAME>/reliefforge.git reliefforge
cd reliefforge
pip -q install PyYAML

In [ ]:
%%bash
cd /kaggle/working/reliefforge
python src/data_download.py --out /kaggle/working/dataset --n-scenes 4000 --seed0 0 --workers 4 --textures

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/reliefforge/src")
from synth.generate import contact_sheet

sheet = contact_sheet("/kaggle/working/dataset", n=24)
print("contact sheet:", sheet)
# LOOK at /kaggle/working/dataset/contact_sheet.png before training (QC gate, PRD_v2 §5.7)

In [ ]:
!du -sh /kaggle/working/dataset && ls /kaggle/working/dataset
!cat /kaggle/working/dataset/scenes.jsonl | head -2

**After Save & Run All finishes:** Output tab → New Dataset (`reliefforge-dataset`),
then in `nb02_train` add it as **Add Input → Your Work → reliefforge-dataset**
(it will appear at `/kaggle/input/reliefforge-dataset`).

Optional (recommended for the course form): publish the dataset folder to the
Hugging Face Hub with `huggingface_hub` (HF token as a Kaggle Secret `HF_TOKEN`).